# 14 Python intermediate - `collections` i `enum`  v2.0

## Rozkład jazdy

1. 🔹 **`Counter` i `defaultdict`** - zliczanie i domyślne wartości
2. 🔹 **`deque` i `namedtuple`** - kolejka i nazwane krotki
3. 🔹 **`enum.Enum` i `IntEnum`** - wyliczenia
4. 🔹 **`enum.auto()` i `Flag`** - zaawansowane wyliczenia
   🐍 Ćwiczenia do każdej sekcji

---

## 1. 🔹 `Counter` i `defaultdict`

`collections.Counter` to słownik zliczający wystąpienia elementów.
Można go tworzyć z dowolnego iterable lub ze słownika.

`collections.defaultdict(factory)` to słownik, który zamiast rzucać
`KeyError` dla brakującego klucza, tworzy wartość domyślną wywołując
`factory()`. Działa jak zwykły `dict` - różni się tylko zachowaniem
przy brakującym kluczu.

| Klasa | Opis | Przykład fabryki |
|---|---|---|
| `Counter` | zliczanie elementów | - |
| `defaultdict(int)` | domyślna wartość 0 | `int` |
| `defaultdict(list)` | domyślna pusta lista | `list` |
| `defaultdict(set)` | domyślny pusty set | `set` |

> 💡 `Counter` wspiera operacje arytmetyczne: `c1 + c2`, `c1 - c2`,
> `c1 & c2` (minimum), `c1 | c2` (maximum).

In [ ]:
from collections import Counter, defaultdict

# Counter
text = 'abracadabra'
c = Counter(text)
print(c)                        # Counter({'a':5,'b':2,'r':2,'c':1,'d':1})
print(c.most_common(3))         # [('a',5),('b',2),('r',2)]
print(c['a'], c['z'])           # 5 0  (nie rzuca KeyError)

words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']
wc = Counter(words)
print(wc.most_common(2))        # [('apple',3),('banana',2)]


# defaultdict - grupowanie
students = [
    ('Alice', 'Math'), ('Bob', 'Physics'),
    ('Carol', 'Math'), ('Dave', 'Physics'), ('Eve', 'Math'),
]
by_subject = defaultdict(list)
for name, subject in students:
    by_subject[subject].append(name)
print(dict(by_subject))

# defaultdict - zliczanie (alternatywa dla Counter)
freq = defaultdict(int)
for ch in 'hello world':
    freq[ch] += 1
print(sorted(freq.items()))

---

### 🐍 Ćwiczenia - Counter i defaultdict

**Ćw. 1.** Dane są dwa teksty. Użyj `Counter`, aby znaleźć:
a) 5 najczęstszych słów w tekście,
b) słowa, które są w obu tekstach (użyj `&`),
c) słowa unikalne dla pierwszego tekstu (użyj `-`).

**Ćw. 2.** Masz listę transakcji `(user, amount)`. Użyj
`defaultdict(list)`, aby pogrupować transakcje per użytkownik,
a następnie oblicz sumę dla każdego użytkownika.

**Ćw. 3.** *(Trudniejsze)* Napisz funkcję `word_frequency(text)`,
która zwraca `Counter` słów (pomija znaki interpunkcyjne, ignoruje
wielkość liter). Następnie znajdź słowa, które pojawiają się
dokładnie raz (hapax legomena).
# hint: użyj str.translate() lub re.sub() do usunięcia interpunkcji

In [ ]:
# Ćw. 1
from collections import Counter

text1 = 'the cat sat on the mat the cat sat'
text2 = 'the dog sat on the log the cat ran'

c1 = Counter(text1.split())
c2 = Counter(text2.split())

top5 = ...
common = ...
unique_to_1 = ...

print('Top 5:', top5)
print('Common:', dict(common))
print('Unique to text1:', dict(unique_to_1))

In [ ]:
# Ćw. 2
from collections import defaultdict

transactions = [
    ('Alice', 00), ('Bob', 50), ('Alice', 200),
    ('Carol', 75),  ('Bob', 30), ('Alice', 50),
]
by_user = defaultdict(list)
# ... grupuj

totals = {}
# ... sumuj

print(totals)  # {'Alice': 350, 'Bob': 80, 'Carol': 75}

In [ ]:
# Ćw. 3
from collections import Counter
import re

def word_frequency(text: str) -> Counter:
    ...


sample = ('To be or not to be, that is the question. '
          'Whether tis nobler in the mind to suffer.')
freq = word_frequency(sample)
print(freq.most_common(5))
hapax = [w for w, c in freq.items() if c == 1]
print('Hapax:', sorted(hapax))

---

## 2. 🔹 `deque`, `namedtuple` i `OrderedDict`

`collections.deque` (double-ended queue) to kolejka dwustronna.
Operacje na obu końcach mają złożoność O(1), w przeciwieństwie
do `list`, gdzie `insert(0,x)` ma O(n).

`collections.namedtuple(name, fields)` tworzy klasę krotek
z nazwanymi polami - czytelniejszą niż zwykła krotka.

`collections.OrderedDict` to słownik pamiętający kolejność
wstawiania (historycznie ważny przed Python 3.7; nadal przydatny
do `move_to_end()` i porównań uwzgledniających kolejność).

In [ ]:
from collections import deque, namedtuple

# deque jako kolejka FIFO
q = deque()
q.append('first')
q.append('second')
q.appendleft('zero')   # dodaj na początku
print(q)               # deque(['zero', 'first', 'second'])
print(q.popleft())     # 'zero' - wyjmij z początku
print(q)

# deque z maxlen - bufor historii
history = deque(maxlen=3)
for cmd in ['ls', 'cd /tmp', 'pwd', 'cat file', 'exit']:
    history.append(cmd)
print(history)   # ostatnie 3 komendy


# namedtuple
Point = namedtuple('Point', ['x', 'y'])
p = Point(3, 4)
print(p.x, p.y)         # 3 4
print(p[0], p[1])       # 3 4  (działa jak krotka)
print(p._asdict())      # {'x': 3, 'y': 4}

# namedtuple z wartościami domyślnymi (Python 3.6.1+)
Color = namedtuple('Color', ['r', 'g', 'b', 'a'], defaults=[255])
red = Color(255, 0, 0)
print(red)    # Color(r=255, g=0, b=0, a=255)

In [ ]:
from collections import OrderedDict

# OrderedDict - slownikipamieta kolejnosc wstawiania
od = OrderedDict()
od['b'] = 2
od['a'] = 1
od['c'] = 3
print(list(od.keys()))   # ['b', 'a', 'c']

# move_to_end - przesuniecie elementu na koniec lub poczatek
od.move_to_end('b')            # przesuń 'b' na koniec
print(list(od.keys()))   # ['a', 'c', 'b']

od.move_to_end('b', last=False)  # przesuń 'b' na poczatek
print(list(od.keys()))   # ['b', 'a', 'c']

# Porownanie kolejnosci (zwykly dict ignoruje kolejnosc)
d1 = OrderedDict([('a', 1), ('b', 2)])
d2 = OrderedDict([('b', 2), ('a', 1)])
print(d1 == d2)   # False - inna kolejnosc
print(dict(d1) == dict(d2))   # True - zwykly dict ignoruje kolejnosc

---

### 🐍 Ćwiczenia - deque, namedtuple i OrderedDict

**Ćw. 4.** Zaimplementuj klasę `Queue` (kolejka FIFO) opierającą
się na `deque`. Powinna obsługiwać: `enqueue(item)`, `dequeue()`,
`peek()`, `is_empty()`, `__len__`.

**Ćw. 5.** Utwórz namedtuple `Employee(name, department, salary)`.
Dla listy pracowników posortuj po pensji (malejąco) i oblicz
srednie wynagrodzenie per wydział używając `defaultdict`.

**Ćw. 6. *(Trudniejsze)*** Zaimplementuj `LRUCache(capacity)` 
z metodami `get(key)` i `put(key, value)` używając `deque` i `dict`.

**Ćw. 7.** Użyj `OrderedDict` do zaimplementowania funkcji
`recent_unique(items, n)` zwracającej `n` ostatnio dodanych
unikalnych elementów w kolejności dodania (najnowszy na końcu).

In [ ]:
# Ćw. 4
from collections import deque

class Queue:
    def __init__(self):
        self._data = deque()

    def enqueue(self, item) -> None:
        ...

    def dequeue(self):
        ...

    def peek(self):
        ...

    def is_empty(self) -> bool:
        ...

    def __len__(self) -> int:
        ...


q = Queue()
q.enqueue(1)
q.enqueue(2)
q.enqueue(3)
print(len(q), q.peek(), q.dequeue(), len(q))

In [ ]:
# Ćw. 5
from collections import namedtuple, defaultdict

Employee = namedtuple('Employee', ['name', 'department', 'salary'])

employees = [
    Employee('Alice', 'Engineering', 9000),
    Employee('Bob',   'Marketing',   7000),
    Employee('Carol', 'Engineering', 8500),
    Employee('Dave',  'Marketing',   6500),
    Employee('Eve',   'Engineering', 9500),
]

# posortuj po wydziale, potem po wynagrodzeniu malejąco
sorted_emp = ...
print(sorted_emp)

# średnie per wydział
avg_salary = ...
print(avg_salary)

In [ ]:
# Ćw. 6
from collections import deque

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache: dict = {}
        self.order: deque = deque()

    def get(self, key: int) -> int:
        ...

    def put(self, key: int, value: int) -> None:
        ...


cache = LRUCache(3)
cache.put(1, 10)
cache.put(2, 20)
cache.put(3, 30)
print(cache.get(1))    # 10
cache.put(4, 40)       # wypchnięcie 2 (najstarszy nieużywany)
print(cache.get(2))    # -1 (usunięty)
print(cache.get(3))    # 30

In [ ]:
# Ćw. 7
from collections import OrderedDict

def recent_unique(items, n):
    # hint: uzyj OrderedDict jako zbioru z kolejnoscia;
    # move_to_end() przy ponownym trafieniu elementu
    seen = OrderedDict()
    for item in items:
        ...
    return list(seen.keys())[-n:]


print(recent_unique([1, 2, 3, 1, 4, 2, 5], 3))   # [1, 4, 5] - nie, powinno byc [4, 2, 5]
print(recent_unique(['a', 'b', 'a', 'c'], 2))    # ['a', 'c']

---

## 3. 🔹 `enum.Enum` i `IntEnum`

Wyliczenie (enumeration) to zbiór stałych symbolicznych.
Python dostarcza kilka typów wyliczeń:

| Klasa | Opis | Przykład |
|---|---|---|
| `Enum` | podstawowe wyliczenie | `Color.RED` |
| `IntEnum` | wyliczenie kompatybilne z `int` | `Status(200)` |
| `StrEnum` (3.11+) | wyliczenie kompatybilne z `str` | `Direction('north')` |
| `Flag` | wyliczenie z operacjami bitowymi | `Permission.READ \| WRITE` |

Zalety wyliczeń nad stałymi:
- nazwy są czytelne i samodokumentujące się,
- nie można pomylić wartości o tym samym typie,
- możliwość iterowania, porównywania, użycia w `match`.

> 💡 Dostęp do elementu: `Color['RED']` (po nazwie) lub
> `Color(1)` (po wartości).

In [ ]:
from enum import Enum, IntEnum


class Color(Enum):
    RED   = 1
    GREEN = 2
    BLUE  = 3


print(Color.RED)           # Color.RED
print(Color.RED.name)      # 'RED'
print(Color.RED.value)     # 1
print(Color['RED'])        # Color.RED  (po nazwie)
print(Color(2))            # Color.GREEN  (po wartości)
print(list(Color))         # wszystkie elementy


class HttpStatus(IntEnum):
    OK        = 200
    NOT_FOUND = 404
    ERROR     = 500


status = HttpStatus.OK
print(status == 200)       # True - IntEnum porównuje z int
print(status > 300)        # False

def handle(code: int):
    if code == HttpStatus.OK:
        print('Success')
    elif code == HttpStatus.NOT_FOUND:
        print('Not found')

handle(200)  # Success

---

### 🐍 Ćwiczenia - Enum i IntEnum

**Ćw. 8.** Utwórz wyliczenie `Direction` z wartościami `NORTH`,
`SOUTH`, `EAST`, `WEST`. Dodaj metodę `opposite()` zwracającą
przeciwny kierunek. Przetestuj dla wszystkich kierunków.

**Ćw. 9.** Utwórz `IntEnum` o nazwie `Priority` z wartościami
`LOW=1`, `MEDIUM=2`, `HIGH=3`, `CRITICAL=4`. Napisz funkcję
`filter_tasks(tasks, min_priority)`, która zwraca zadania
o priorytecie >= min_priority.

**Ćw. 10.** *(Trudniejsze)* Utwórz wyliczenie `Planet` (jak
w dokumentacji Pythona) z polem `mass` i `radius`. Dodaj
właściwość `surface_gravity` i metodę `weight(mass_on_earth)`.
# hint: g = G * mass / radius**2, G = 6.67430e-11

In [ ]:
# Ćw. 8
from enum import Enum

class Direction(Enum):
    NORTH = 'N'
    SOUTH = 'S'
    EAST  = 'E'
    WEST  = 'W'

    def opposite(self) -> 'Direction':
        ...


for d in Direction:
    print(f'{d.name} -> {d.opposite().name}')

In [ ]:
# Ćw. 9
from enum import IntEnum

class Priority(IntEnum):
    LOW      = 1
    MEDIUM   = 2
    HIGH     = 3
    CRITICAL = 4


def filter_tasks(tasks: list, min_priority: Priority) -> list:
    ...


tasks = [
    ('Write tests',    Priority.HIGH),
    ('Update docs',    Priority.LOW),
    ('Fix prod bug',   Priority.CRITICAL),
    ('Code review',    Priority.MEDIUM),
]
print(filter_tasks(tasks, Priority.HIGH))

In [ ]:
# Ćw. 10
from enum import Enum

G = 6.67430e-11

class Planet(Enum):
    MERCURY = (3.303e+23, 2.4397e6)
    VENUS   = (4.869e+24, 6.0518e6)
    EARTH   = (5.976e+24, 6.37814e6)
    MARS    = (6.421e+23, 3.3972e6)

    def __init__(self, mass: float, radius: float):
        ...

    @property
    def surface_gravity(self) -> float:
        ...

    def weight(self, mass_on_earth: float) -> float:
        ...


for planet in Planet:
    print(f'{planet.name}: {planet.weight(75):.2f} N')

---

## 4. 🔹 `enum.auto()` i `Flag`

`enum.auto()` automatycznie przypisuje wartości kolejnym elementom.
Domyślnie numeruje od 1, ale można to zmienić przez `_generate_next_value_`.

`enum.Flag` to wyliczenie z operacjami bitowymi. Pozwala kombinować
wartości za pomocą `|` (OR), `&` (AND), `~` (NOT). Idealny dla
flag uprawnień, opcji, stanów.

| Operator | Znaczenie | Przykład |
|---|---|---|
| `A \| B` | A lub B (oba) | `READ \| WRITE` |
| `A & B` | tylko część wspólna | `perms & READ` |
| `A in B` | czy A zawiera się w B | `READ in perms` |

In [ ]:
from enum import Enum, Flag, auto


class Color(Enum):
    RED   = auto()   # 1
    GREEN = auto()   # 2
    BLUE  = auto()   # 3

print(list(Color))   # [Color.RED, Color.GREEN, Color.BLUE]


class Permission(Flag):
    NONE    = 0
    READ    = auto()   # 1
    WRITE   = auto()   # 2
    EXECUTE = auto()   # 4
    ALL     = READ | WRITE | EXECUTE


user_perms = Permission.READ | Permission.WRITE
print(user_perms)                          # Permission.READ|WRITE
print(Permission.READ in user_perms)       # True
print(Permission.EXECUTE in user_perms)    # False

admin_perms = Permission.ALL
print(admin_perms & user_perms)            # Permission.READ|WRITE

---

### 🐍 Ćwiczenia - auto() i Flag

**Ćw. 11.** Utwórz wyliczenie `Weekday` używając `auto()`. Dodaj
metodę `is_weekend()` zwracającą `True` dla soboty i niedzieli.
Wypisz wszystkie dni tygodnia z informacją o weekendzie.

**Ćw. 12.** Utwórz `Flag` o nazwie `FileMode` z flagami `READ`,
`WRITE`, `APPEND`, `BINARY`. Napisz funkcję `open_mode(flags)`,
która zwraca string trybu do `open()`, np. `'rb'` dla
`READ | BINARY`.

**Ćw. 13.** *(Trudniejsze)* Utwórz wyliczenie `Status` jako `Enum`
ze stanami zamówienia: `PENDING`, `PROCESSING`, `SHIPPED`,
`DELIVERED`, `CANCELLED`. Dodaj metodę `can_transition_to(next)`,
która sprawdza, czy przejście między stanami jest dozwolone
(zdefiniuj dozwolone przejścia jako słownik w klasie).

In [ ]:
# Ćw. 11
from enum import Enum, auto

class Weekday(Enum):
    MONDAY    = auto()
    TUESDAY   = auto()
    WEDNESDAY = auto()
    THURSDAY  = auto()
    FRIDAY    = auto()
    SATURDAY  = auto()
    SUNDAY    = auto()

    def is_weekend(self) -> bool:
        ...


for day in Weekday:
    weekend = '(weekend)' if day.is_weekend() else ''
    print(f'{day.name} {weekend}')

In [ ]:
# Ćw. 12
from enum import Flag, auto

class FileMode(Flag):
    READ   = auto()
    WRITE  = auto()
    APPEND = auto()
    BINARY = auto()


def open_mode(flags: FileMode) -> str:
    ...


print(open_mode(FileMode.READ))                    # 'r'
print(open_mode(FileMode.READ | FileMode.BINARY))  # 'rb'
print(open_mode(FileMode.WRITE))                   # 'w'
print(open_mode(FileMode.APPEND | FileMode.BINARY))# 'ab'

In [ ]:
# Ćw. 13
from enum import Enum, auto

class Status(Enum):
    PENDING    = auto()
    PROCESSING = auto()
    SHIPPED    = auto()
    DELIVERED  = auto()
    CANCELLED  = auto()

    # dozwolone przejścia: {stan: {możliwe następne stany}}
    TRANSITIONS = {
        # uzupełnij
    }

    def can_transition_to(self, next_status: 'Status') -> bool:
        ...


s = Status.PENDING
print(s.can_transition_to(Status.PROCESSING))   # True
print(s.can_transition_to(Status.DELIVERED))    # False
print(s.can_transition_to(Status.CANCELLED))    # True